In [ ]:
# Agents: 
# Plan → Act → Observe

# Tools extend what agents can do—letting them fetch real-time data, execute code, query external databases, and take actions in the world.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import os
import httpx

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model


# ============================================================
# Load Environment Variables
# ============================================================

load_dotenv()

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

if not OPENWEATHER_API_KEY:
    raise ValueError(
        "OPENWEATHER_API_KEY is not configured. "
        "Add it to your .env file."
    )


# ============================================================
# Math Tools
# ============================================================

@tool
def addition(a: int = 0, b: int = 0):
    """Return the addition of two numbers."""
    return a + b


@tool
def multiplication(a: int = 1, b: int = 1):
    """Return the multiplication of two numbers."""
    return a * b


# ============================================================
# Weather Tool
# ============================================================

@tool
async def get_weather(location: str):
    """
    Get the current weather for a given city/location
    using the OpenWeather API.
    """

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": location,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",
    }

    try:

        async with httpx.AsyncClient(timeout=10) as client:

            response = await client.get(
                url,
                params=params,
            )

        # OpenWeather error
        if response.status_code != 200:

            try:
                error_data = response.json()
                message = error_data.get(
                    "message",
                    "Unknown error"
                )

            except Exception:
                message = response.text

            return f"Could not get weather for {location}: {message}"

        data = response.json()

        weather = data["weather"][0]

        return {
            "location": data["name"],
            "country": data["sys"]["country"],
            "temperature_celsius": data["main"]["temp"],
            "feels_like_celsius": data["main"]["feels_like"],
            "humidity_percent": data["main"]["humidity"],
            "description": weather["description"],
            "wind_speed_m_s": data["wind"]["speed"],
        }

    except httpx.RequestError as e:

        return f"Weather API request failed: {str(e)}"


# ============================================================
# LLM
# ============================================================

llm = init_chat_model(
    model="gemini-3.5-flash-lite",
    model_provider="google_genai",
)


# ============================================================
# Agent
# ============================================================

agent = create_agent(
    model=llm,
    tools=[
        addition,
        multiplication,
        get_weather,
    ],
)


# ============================================================
# Test Weather Tool Directly
# ============================================================

# weather = await get_weather.ainvoke({
#     "location": "Agartala"
# })

# print("Direct weather tool result:")
# print(weather)


# ============================================================
# Ask Agent
# ============================================================

result = await agent.ainvoke({
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. "
                "Use the available tools when necessary. "
                "For weather questions, always use the get_weather tool. "
                "Give the user a clear and concise answer."
            ),
        },
        {
            "role": "user",
            "content": "What's the weather like in Agartala?",
        },
    ],
})


# ============================================================
# Final Answer
# ============================================================

print("\nAgent response:")
print(result["messages"][-1].content)

Direct weather tool result:
{'location': 'Agartala', 'country': 'IN', 'temperature_celsius': 34.01, 'feels_like_celsius': 41.01, 'humidity_percent': 62, 'description': 'heavy intensity rain', 'wind_speed_m_s': 3.6}

Agent response:
[{'type': 'text', 'text': 'The current weather in Agartala features heavy intensity rain with a temperature of 34.01°C (feels like 41.01°C), a humidity of 62%, and a wind speed of 3.6 m/s.', 'extras': {'signature': 'El4KXAERTTIPNhLOUGl5d4BLxoQOEky0ONGYbdSvufBnk65ZD1NGVe+h5/NwjBN+8uXctzIH1TqTq9GcJpTqOZh+ND35OtNQ3rfuEO3wihkd92HUnGFYxb0tjZrsjytb'}}]
